# 04_gold — Star schema for reporting

Gold is modelled as a **star schema**: fact tables hold the business events and their numbers; dimension tables describe who, what, and when. Power BI and Synapse read these tables.

```
               dim_date
                  |
dim_partner -- fact_bank_payment
dim_partner -- fact_tpa_settlement -- dim_provider
dim_partner -- fact_repair_invoice -- dim_repair_shop
dim_partner -- fact_data_quality_issue
                  |
               dim_date
```

| Table | Type | Grain (one row per) |
|---|---|---|
| `dim_date` | Dimension | Calendar day, 2025-01-01 to 2027-12-31, plus an Unknown row |
| `dim_partner` | Dimension | Partner feed (BANK, TPA, REPAIR) |
| `dim_provider` | Dimension | TPA provider, plus an Unknown row |
| `dim_repair_shop` | Dimension | Repair shop, plus an Unknown row |
| `fact_bank_payment` | Fact | Valid bank payment |
| `fact_tpa_settlement` | Fact | Valid TPA settlement |
| `fact_repair_invoice` | Fact | Valid repair invoice |
| `fact_data_quality_issue` | Fact | Error or warning on a Silver record |

**Keys**
- `date_key` is the date as a number in `yyyyMMdd` format, for example `20261012`.
- `provider_key` and `shop_key` are a hash of the name (`xxhash64`), so the same name always gets the same key on every rebuild.
- `-1` is the Unknown member, used when a value is missing.
- Business references (`bank_ref`, `tpa_ref`, `invoice_no`) stay in the facts as degenerate dimensions.

**Rules**
- Payment, settlement, and invoice facts contain `VALID` Silver rows only. Invalid rows are reported in `fact_data_quality_issue`.
- Every table is rebuilt from Silver on each run with `CREATE OR REPLACE`.
- Tables are stored at `abfss://lakehouse@insuranceadls26.dfs.core.windows.net/gold/<table>` so Synapse serverless SQL can read the Delta files directly. They are external tables; dropping a table does not delete its files.

## Step 0: Remove the earlier summary tables

**Why:** an earlier version of this notebook created summary tables. They are replaced by the star schema. Skip this step if the earlier version was never run.

In [0]:
%sql
DROP TABLE IF EXISTS insurance.gold.partner_daily_summary;
DROP TABLE IF EXISTS insurance.gold.bank_payments_monthly;
DROP TABLE IF EXISTS insurance.gold.tpa_provider_monthly;
DROP TABLE IF EXISTS insurance.gold.repair_shop_monthly;
DROP TABLE IF EXISTS insurance.gold.data_quality_issues;

## Step 1: Date dimension (`dim_date`)

**Why:** Reports filter and group by year, quarter, month, and day name. A date dimension holds these attributes once, so every fact can use them through `date_key`. The calendar is generated with `sequence` and `explode`, one row per day.

In [0]:
%sql
CREATE OR REPLACE TABLE insurance.gold.dim_date
LOCATION 'abfss://lakehouse@insuranceadlsgenk2.dfs.core.windows.net/gold/dim_date'
AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d                                       AS full_date,
  year(d)                                 AS year,
  quarter(d)                              AS quarter,
  month(d)                                AS month,
  date_format(d, 'MMMM')                  AS month_name,
  date_format(d, 'yyyy-MM')               AS year_month,
  day(d)                                  AS day_of_month,
  date_format(d, 'EEEE')                  AS day_name,
  dayofweek(d) IN (1, 7)                  AS is_weekend
FROM (SELECT explode(sequence(DATE'2025-01-01', DATE'2027-12-31', INTERVAL 1 DAY)) AS d)
UNION ALL
SELECT -1, NULL, NULL, NULL, NULL, 'Unknown', 'Unknown', NULL, 'Unknown', NULL;

**Expected:** 1,096 days plus the Unknown row (1,097 rows). `20261011` is a Sunday, so `is_weekend` is true.

In [0]:
%sql
SELECT * FROM insurance.gold.dim_date ORDER BY 1 LIMIT 20;

## Step 2: Partner dimension (`dim_partner`)

**Why:** One row per partner feed with a readable name. The facts store only `partner_key`.

In [0]:
%sql
CREATE OR REPLACE TABLE insurance.gold.dim_partner
LOCATION 'abfss://lakehouse@insuranceadlsgenk2.dfs.core.windows.net/gold/dim_partner'
AS
SELECT * FROM VALUES
  (1, 'BANK',   'Bank',                      'Payment confirmations'),
  (2, 'TPA',    'Third-party administrator', 'Medical claim settlements'),
  (3, 'REPAIR', 'Repair shops',              'Auto repair invoices')
AS t(partner_key, source_system, partner_name, feed_description);

**Expected:** 3 rows: 1 BANK, 2 TPA, 3 REPAIR.

In [0]:
%sql
SELECT * FROM insurance.gold.dim_partner ORDER BY 1 LIMIT 20;

## Step 3: Provider dimension (`dim_provider`)

**Why:** One row per distinct provider name from Silver. Silver already standardised the names, so `Memorial Hermann` appears once.

In [0]:
%sql
CREATE OR REPLACE TABLE insurance.gold.dim_provider
LOCATION 'abfss://lakehouse@insuranceadlsgenk2.dfs.core.windows.net/gold/dim_provider'
AS
SELECT DISTINCT
  xxhash64(provider_name) AS provider_key,
  provider_name
FROM insurance.silver.tpa_settlement
WHERE provider_name IS NOT NULL
UNION ALL
SELECT -1, 'Unknown';

**Expected:** 6 rows: 5 providers plus Unknown.

In [0]:
%sql
SELECT * FROM insurance.gold.dim_provider ORDER BY 1 LIMIT 20;

## Step 4: Repair shop dimension (`dim_repair_shop`)

**Why:** One row per distinct repair shop name from Silver.

In [0]:
%sql
CREATE OR REPLACE TABLE insurance.gold.dim_repair_shop
LOCATION 'abfss://lakehouse@insuranceadlsgenk2.dfs.core.windows.net/gold/dim_repair_shop'
AS
SELECT DISTINCT
  xxhash64(shop_name) AS shop_key,
  shop_name
FROM insurance.silver.repair_invoice
WHERE shop_name IS NOT NULL
UNION ALL
SELECT -1, 'Unknown';

**Expected:** 6 rows: 5 shops plus Unknown.

In [0]:
%sql
SELECT * FROM insurance.gold.dim_repair_shop ORDER BY 1 LIMIT 20;

## Step 5: Bank payment fact (`fact_bank_payment`)

**Why:** One row per valid bank payment. `cleared_amount` and `returned_amount` split the amount by status so reports can sum each directly.

In [0]:
%sql
CREATE OR REPLACE TABLE insurance.gold.fact_bank_payment
LOCATION 'abfss://lakehouse@insuranceadlsgenk2.dfs.core.windows.net/gold/fact_bank_payment'
AS
SELECT
  b.bank_ref,
  b.payment_id,
  b.claim_id,
  p.partner_key,
  COALESCE(CAST(date_format(b.settlement_date, 'yyyyMMdd') AS INT), -1) AS settlement_date_key,
  CAST(date_format(b.file_date, 'yyyyMMdd') AS INT)                     AS file_date_key,
  b.status,
  b.amount,
  CASE WHEN b.status = 'CLEARED'  THEN b.amount ELSE 0 END              AS cleared_amount,
  CASE WHEN b.status = 'RETURNED' THEN b.amount ELSE 0 END              AS returned_amount,
  b.record_versions,
  current_timestamp()                                                   AS _gold_loaded_at
FROM insurance.silver.bank_settlement b
JOIN insurance.gold.dim_partner p ON p.source_system = 'BANK'
WHERE b.dq_status = 'VALID';

**Expected:** 31 rows, total amount 50317.27. `BNK202610110003` has status RETURNED and `record_versions` 3.

In [0]:
%sql
SELECT * FROM insurance.gold.fact_bank_payment ORDER BY 1 LIMIT 20;

## Step 6: TPA settlement fact (`fact_tpa_settlement`)

**Why:** One row per valid TPA settlement, linked to the provider and the service date.

In [0]:
%sql
CREATE OR REPLACE TABLE insurance.gold.fact_tpa_settlement
LOCATION 'abfss://lakehouse@insuranceadlsgenk2.dfs.core.windows.net/gold/fact_tpa_settlement'
AS
SELECT
  t.tpa_ref,
  t.claim_id,
  p.partner_key,
  COALESCE(xxhash64(t.provider_name), -1)                              AS provider_key,
  COALESCE(CAST(date_format(t.service_date, 'yyyyMMdd') AS INT), -1)   AS service_date_key,
  CAST(date_format(t.file_date, 'yyyyMMdd') AS INT)                    AS file_date_key,
  t.billed_amount,
  t.approved_amount,
  t.amount_not_covered,
  t.record_versions,
  current_timestamp()                                                  AS _gold_loaded_at
FROM insurance.silver.tpa_settlement t
JOIN insurance.gold.dim_partner p ON p.source_system = 'TPA'
WHERE t.dq_status = 'VALID';

**Expected:** 24 rows, total approved amount 58418.66.

In [0]:
%sql
SELECT * FROM insurance.gold.fact_tpa_settlement ORDER BY 1 LIMIT 20;

## Step 7: Repair invoice fact (`fact_repair_invoice`)

**Why:** One row per valid repair invoice, linked to the shop and the invoice date.

In [0]:
%sql
CREATE OR REPLACE TABLE insurance.gold.fact_repair_invoice
LOCATION 'abfss://lakehouse@insuranceadlsgenk2.dfs.core.windows.net/gold/fact_repair_invoice'
AS
SELECT
  r.invoice_no,
  r.claim_id,
  p.partner_key,
  COALESCE(xxhash64(r.shop_name), -1)                                  AS shop_key,
  COALESCE(CAST(date_format(r.invoice_date, 'yyyyMMdd') AS INT), -1)   AS invoice_date_key,
  CAST(date_format(r.file_date, 'yyyyMMdd') AS INT)                    AS file_date_key,
  r.labor_amount,
  r.parts_amount,
  r.total_amount,
  r.record_versions,
  current_timestamp()                                                  AS _gold_loaded_at
FROM insurance.silver.repair_invoice r
JOIN insurance.gold.dim_partner p ON p.source_system = 'REPAIR'
WHERE r.dq_status = 'VALID';

**Expected:** 21 rows, total amount 52702.97.

In [0]:
%sql
SELECT * FROM insurance.gold.fact_repair_invoice ORDER BY 1 LIMIT 20;

## Step 8: Data quality issue fact (`fact_data_quality_issue`)

**Why:** One row per error or warning on a Silver record. `explode` turns each code in `dq_errors` and `dq_warnings` into its own row. The operations team uses this table to follow up with partners.

In [0]:
%sql
CREATE OR REPLACE TABLE insurance.gold.fact_data_quality_issue
LOCATION 'abfss://lakehouse@insuranceadlsgenk2.dfs.core.windows.net/gold/fact_data_quality_issue'
AS
SELECT
  p.partner_key,
  i.record_ref,
  CAST(date_format(i.file_date, 'yyyyMMdd') AS INT) AS file_date_key,
  i.severity,
  i.issue_code,
  i._source_file,
  current_timestamp()                               AS _gold_loaded_at
FROM (
  SELECT 'BANK'   AS source_system, bank_ref   AS record_ref, file_date, _source_file, 'ERROR'   AS severity, explode(dq_errors)   AS issue_code FROM insurance.silver.bank_settlement
  UNION ALL
  SELECT 'TPA',    tpa_ref,    file_date, _source_file, 'ERROR',   explode(dq_errors)   FROM insurance.silver.tpa_settlement
  UNION ALL
  SELECT 'TPA',    tpa_ref,    file_date, _source_file, 'WARNING', explode(dq_warnings) FROM insurance.silver.tpa_settlement
  UNION ALL
  SELECT 'REPAIR', invoice_no, file_date, _source_file, 'ERROR',   explode(dq_errors)   FROM insurance.silver.repair_invoice
  UNION ALL
  SELECT 'REPAIR', invoice_no, file_date, _source_file, 'WARNING', explode(dq_warnings) FROM insurance.silver.repair_invoice
) i
JOIN insurance.gold.dim_partner p ON p.source_system = i.source_system;

**Expected:** 15 rows: 13 errors and 2 warnings.

In [0]:
%sql
SELECT * FROM insurance.gold.fact_data_quality_issue ORDER BY 1 LIMIT 20;

## Step 9: Facts match Silver

**Why:** proves every valid Silver record reached Gold once, and no amount changed. `gold_rows` must equal `silver_valid_rows`, and `gold_amount` must equal `silver_amount`.

**Expected:** 31 / 24 / 21 rows; 50317.27 / 58418.66 / 52702.97.

In [0]:
%sql
SELECT 'fact_bank_payment' AS fact_table,
       (SELECT COUNT(*) FROM insurance.gold.fact_bank_payment)                                AS gold_rows,
       (SELECT COUNT(*) FROM insurance.silver.bank_settlement WHERE dq_status = 'VALID')       AS silver_valid_rows,
       (SELECT SUM(amount) FROM insurance.gold.fact_bank_payment)                             AS gold_amount,
       (SELECT SUM(amount) FROM insurance.silver.bank_settlement WHERE dq_status = 'VALID')    AS silver_amount
UNION ALL
SELECT 'fact_tpa_settlement',
       (SELECT COUNT(*) FROM insurance.gold.fact_tpa_settlement),
       (SELECT COUNT(*) FROM insurance.silver.tpa_settlement WHERE dq_status = 'VALID'),
       (SELECT SUM(approved_amount) FROM insurance.gold.fact_tpa_settlement),
       (SELECT SUM(approved_amount) FROM insurance.silver.tpa_settlement WHERE dq_status = 'VALID')
UNION ALL
SELECT 'fact_repair_invoice',
       (SELECT COUNT(*) FROM insurance.gold.fact_repair_invoice),
       (SELECT COUNT(*) FROM insurance.silver.repair_invoice WHERE dq_status = 'VALID'),
       (SELECT SUM(total_amount) FROM insurance.gold.fact_repair_invoice),
       (SELECT SUM(total_amount) FROM insurance.silver.repair_invoice WHERE dq_status = 'VALID')
;

## Step 10: Every fact key exists in its dimension

**Why:** a fact row whose key is missing from the dimension disappears from reports that join to that dimension. `LEFT ANTI JOIN` returns fact rows with no matching dimension row.

**Expected:** `orphan_rows` is 0 on every row.

In [0]:
%sql
SELECT 'fact_bank_payment -> dim_date (settlement)' AS relationship,
       COUNT(*) AS orphan_rows
FROM insurance.gold.fact_bank_payment f
LEFT ANTI JOIN insurance.gold.dim_date d ON f.settlement_date_key = d.date_key
UNION ALL
SELECT 'fact_bank_payment -> dim_partner', COUNT(*)
FROM insurance.gold.fact_bank_payment f
LEFT ANTI JOIN insurance.gold.dim_partner d ON f.partner_key = d.partner_key
UNION ALL
SELECT 'fact_tpa_settlement -> dim_provider', COUNT(*)
FROM insurance.gold.fact_tpa_settlement f
LEFT ANTI JOIN insurance.gold.dim_provider d ON f.provider_key = d.provider_key
UNION ALL
SELECT 'fact_tpa_settlement -> dim_date (service)', COUNT(*)
FROM insurance.gold.fact_tpa_settlement f
LEFT ANTI JOIN insurance.gold.dim_date d ON f.service_date_key = d.date_key
UNION ALL
SELECT 'fact_repair_invoice -> dim_repair_shop', COUNT(*)
FROM insurance.gold.fact_repair_invoice f
LEFT ANTI JOIN insurance.gold.dim_repair_shop d ON f.shop_key = d.shop_key
UNION ALL
SELECT 'fact_repair_invoice -> dim_date (invoice)', COUNT(*)
FROM insurance.gold.fact_repair_invoice f
LEFT ANTI JOIN insurance.gold.dim_date d ON f.invoice_date_key = d.date_key
UNION ALL
SELECT 'fact_data_quality_issue -> dim_date (file)', COUNT(*)
FROM insurance.gold.fact_data_quality_issue f
LEFT ANTI JOIN insurance.gold.dim_date d ON f.file_date_key = d.date_key
;

## Step 11: Dimension keys are unique

**Why:** a duplicate key in a dimension doubles the matching fact rows in every join.

**Expected:** `duplicate_keys` is 0 for every dimension.

In [0]:
%sql
SELECT 'dim_date' AS dimension, COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_keys FROM insurance.gold.dim_date
UNION ALL SELECT 'dim_partner', COUNT(*) - COUNT(DISTINCT partner_key) FROM insurance.gold.dim_partner
UNION ALL SELECT 'dim_provider', COUNT(*) - COUNT(DISTINCT provider_key) FROM insurance.gold.dim_provider
UNION ALL SELECT 'dim_repair_shop', COUNT(*) - COUNT(DISTINCT shop_key) FROM insurance.gold.dim_repair_shop
;

## Step 12: Query the star schema

**Why:** shows how reports use the model: join a fact to its dimensions, then group by dimension attributes.

TPA approved amount and approval rate by month and provider. **Expected:** 5 rows for 2026-10; Memorial Hermann approval rate 0.8466.

In [0]:
%sql
SELECT d.year_month,
       pr.provider_name,
       COUNT(*)                                             AS settlements,
       SUM(f.billed_amount)                                 AS billed_amount,
       SUM(f.approved_amount)                               AS approved_amount,
       ROUND(SUM(f.approved_amount) / SUM(f.billed_amount), 4) AS approval_rate
FROM insurance.gold.fact_tpa_settlement f
JOIN insurance.gold.dim_provider pr ON f.provider_key = pr.provider_key
JOIN insurance.gold.dim_date d      ON f.service_date_key = d.date_key
GROUP BY d.year_month, pr.provider_name
ORDER BY d.year_month, pr.provider_name
;

Bank cleared and returned amounts by month. **Expected:** 2026-10: 31 payments, cleared 40650.49, returned 9666.78, return rate 19.4%.

In [0]:
%sql
SELECT d.year_month,
       COUNT(*)                      AS payments,
       SUM(f.cleared_amount)         AS cleared_amount,
       SUM(f.returned_amount)        AS returned_amount,
       ROUND(100.0 * SUM(CASE WHEN f.status = 'RETURNED' THEN 1 ELSE 0 END) / COUNT(*), 1) AS return_rate_pct
FROM insurance.gold.fact_bank_payment f
JOIN insurance.gold.dim_date d ON f.settlement_date_key = d.date_key
GROUP BY d.year_month
ORDER BY d.year_month
;

Data quality issues by partner and file date. **Expected:** 15 rows.

In [0]:
%sql
SELECT p.partner_name,
       d.full_date                AS file_date,
       f.severity,
       f.issue_code,
       COUNT(*)                   AS records
FROM insurance.gold.fact_data_quality_issue f
JOIN insurance.gold.dim_partner p ON f.partner_key = p.partner_key
JOIN insurance.gold.dim_date d    ON f.file_date_key = d.date_key
GROUP BY p.partner_name, d.full_date, f.severity, f.issue_code
ORDER BY p.partner_name, d.full_date, f.severity, f.issue_code
;

## Step 13: Confirm the storage location

**Why:** Synapse reads the Gold files from this path. **Expected:** `format` is `delta` and `location` ends with `/gold/fact_bank_payment`.

In [0]:
%sql
DESCRIBE DETAIL insurance.gold.fact_bank_payment;